# Comparison of discrete units, continuous embeddings and the CNN

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, f1_score

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "layer3_reliability.py").exists():
        sys.path.insert(0, str(_cand))
        break
import layer3_reliability as rel
import crossfarm_nested as cn

OUT = cn.OUT
OUT_DIR = OUT / "17n_representation_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGETS = ["is_nonvocal", "is_distress_related", "is_resource_related", "proxy_functional_v1"]
COVERAGES = [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
N_BOOT, SEED = 1000, 42
RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

SOURCES = {
    "discrete_units": (OUT / "14n_layer2_crossfarm_nested" / "crossfarm_fold_predictions.csv.gz",
                       OUT / "19n_layer3_reliability_analysis" / "framework_inner_predictions.csv.gz",
                       "target_name", "heldout_farm"),
    "continuous_embeddings": (OUT / "14c_layer2_continuous_embeddings" / "continuous_fold_predictions.csv.gz",
                              OUT / "14c_layer2_continuous_embeddings" / "continuous_inner_predictions.csv.gz",
                              "target_name", "heldout_farm"),
    "cnn_logmel": (OUT / "cnn_baseline" / "cnn_lofo_predictions.csv.gz",
                   OUT / "cnn_baseline" / "cnn_lofo_inner_predictions.csv.gz",
                   "target", "held_farm"),
}
MODELS = list(SOURCES)

outer, l3 = {}, {}
for name, (po, pi, tcol, hcol) in SOURCES.items():
    o = rel.standardise_predictions(pd.read_csv(po), tcol, hcol)
    i = rel.standardise_predictions(pd.read_csv(pi), tcol, hcol, "inner_val_farm")
    outer[name] = o
    l3[name] = rel.summarise_model(o, i, TARGETS, COVERAGES)

for t in TARGETS:
    keys = [set(outer[m].loc[outer[m].target == t, "_record_key"]) for m in MODELS]
    assert all(k == keys[0] for k in keys) and len(keys[0]) == 559, t
print("same 559 clips per target for all models")

In [ ]:
units_pred = pd.read_csv(SOURCES["discrete_units"][0])


def aligned(model, target):
    o = outer[model]
    o = o[o.target == target].sort_values("_record_key").reset_index(drop=True)
    P, _ = rel.prediction_matrix(o)
    return o["_record_key"].to_numpy(), o["y"].to_numpy(), P.argmax(1), o["held"].astype(int).to_numpy()


def mf1(y, p):
    return float(f1_score(y, p, average="macro", zero_division=0))


rows, farm_rows = [], []
for t in TARGETS:
    u = units_pred[units_pred.target_name == t]
    floor_train = mf1(u["y_true"], u["y_dummy"])
    for m in MODELS:
        _, y, p, farms = aligned(m, t)
        rows.append({"model": m, "target": t, "macro_f1": mf1(y, p),
                     "balanced_accuracy": float(balanced_accuracy_score(y, p)),
                     "training_majority_floor_macro_f1": floor_train})
        for f in (1, 2, 3):
            k = farms == f
            uf = u[u.heldout_farm.astype(int) == f]
            own = np.full(k.sum(), pd.Series(y[k]).mode().iloc[0])
            farm_rows.append({"model": m, "target": t, "heldout_farm": f, "n_clips": int(k.sum()),
                              "macro_f1": mf1(y[k], p[k]),
                              "balanced_accuracy": float(balanced_accuracy_score(y[k], p[k])),
                              "training_majority_floor_macro_f1": mf1(uf["y_true"], uf["y_dummy"]),
                              "own_majority_floor_macro_f1": mf1(y[k], own)})
discrimination_df = pd.DataFrame(rows)
perfarm_df = pd.DataFrame(farm_rows)
discrimination_df.pivot(index="target", columns="model", values="macro_f1").loc[TARGETS].round(3)

In [ ]:
PAIRS = [("continuous_embeddings", "discrete_units"), ("cnn_logmel", "discrete_units"),
         ("continuous_embeddings", "cnn_logmel")]
diff_rows = []
for t in TARGETS:
    data = {m: aligned(m, t) for m in MODELS}
    ids = data[MODELS[0]][0]
    assert all((data[m][0] == ids).all() for m in MODELS)
    y = data[MODELS[0]][1]
    rng = np.random.default_rng(SEED)
    idx_all = [rng.integers(0, len(y), len(y)) for _ in range(N_BOOT)]
    for a, b in PAIRS:
        pa, pb = data[a][2], data[b][2]
        boots = np.array([mf1(y[ix], pa[ix]) - mf1(y[ix], pb[ix]) for ix in idx_all])
        farms = data[a][3]
        diff_rows.append({"target": t, "model_a": a, "model_b": b,
                          "macro_f1_difference": mf1(y, pa) - mf1(y, pb),
                          "ci_low": float(np.quantile(boots, 0.025)), "ci_high": float(np.quantile(boots, 0.975)),
                          "share_boot_a_better": float((boots > 0).mean()),
                          **{f"farm{f}_difference": mf1(y[farms == f], pa[farms == f]) - mf1(y[farms == f], pb[farms == f])
                             for f in (1, 2, 3)}})
differences_df = pd.DataFrame(diff_rows)
differences_df.round(3)

In [ ]:
rel_rows, curve_rows, fold_rows = [], [], []
for m in MODELS:
    store, calib, select, curves, folds, pooled = l3[m]
    c, s = calib.set_index("target"), select.set_index("target")
    for t in TARGETS:
        rel_rows.append({"model": m, "target": t,
                         "ece_raw": c.loc[t, "ece_raw"], "ece_nested_calibrated": c.loc[t, "ece_selected"],
                         "brier_raw": c.loc[t, "brier_raw"], "brier_nested_calibrated": c.loc[t, "brier_selected"],
                         "calibration_method": c.loc[t, "selected_method"],
                         "selective_score": s.loc[t, "score_version"],
                         "macro_f1_all_clips": s.loc[t, "baseline_macro_f1"],
                         "retained_macro_f1_80cov": s.loc[t, "retained_macro_f1_80cov"],
                         "selective_gain": s.loc[t, "selective_gain"],
                         "learned_threshold_coverage": s.loc[t, "learned_threshold_coverage"],
                         "learned_threshold_retained_macro_f1": s.loc[t, "learned_threshold_retained_macro_f1"],
                         "learned_threshold_coverage_by_farm": s.loc[t, "learned_threshold_coverage_by_farm"]})
    curve_rows.append(curves.assign(model=m))
    fold_rows.append(folds.assign(model=m))
reliability_df = pd.DataFrame(rel_rows)
curves_df = pd.concat(curve_rows, ignore_index=True)
folds_df = pd.concat(fold_rows, ignore_index=True)
reliability_df.round(3)

In [ ]:
man = pd.read_csv(OUT / "00_manifest" / "manifest.csv").set_index("clip_id")
session_of = man[["farm", "barn_zone", "microphone", "mic_placement_context"]].astype(str).agg(" | ".join, axis=1)
N_CLUSTER_BOOT = 2000


def cluster_indices(ids, farms, rng):
    sessions = session_of.reindex(ids).to_numpy()
    out = []
    for f in np.unique(farms):
        groups = np.unique(sessions[farms == f])
        pick = rng.choice(groups, size=len(groups), replace=True)
        for g in pick:
            out.append(np.flatnonzero((sessions == g) & (farms == f)))
    return np.concatenate(out)


def holm(pvals):
    order = np.argsort(pvals)
    adj = np.empty(len(pvals))
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(pvals) - rank) * pvals[i])
        adj[i] = min(running, 1.0)
    return adj


cl_rows = []
for t in TARGETS:
    data = {m: aligned(m, t) for m in MODELS}
    ids, y, farms = data[MODELS[0]][0], data[MODELS[0]][1], data[MODELS[0]][3]
    rng = np.random.default_rng(SEED)
    idx_all = [cluster_indices(ids, farms, rng) for _ in range(N_CLUSTER_BOOT)]
    for a, b in PAIRS:
        pa, pb = data[a][2], data[b][2]
        boots = np.array([mf1(y[ix], pa[ix]) - mf1(y[ix], pb[ix]) for ix in idx_all])
        p_two = min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean()))
        cl_rows.append({"target": t, "model_a": a, "model_b": b, "macro_f1_difference": mf1(y, pa) - mf1(y, pb),
                        "cluster_ci_low": float(np.quantile(boots, 0.025)),
                        "cluster_ci_high": float(np.quantile(boots, 0.975)), "p_two_sided": p_two})
cluster_df = pd.DataFrame(cl_rows)
cluster_df["p_holm"] = holm(cluster_df["p_two_sided"].to_numpy())

ece_rows = []
for m in MODELS:
    store = l3[m][0]
    for t in TARGETS:
        r = store[t]
        n = len(r["classes"])
        rng = np.random.default_rng(SEED)
        ece = lambda P, ix: rel.expected_calibration_error(r["y"][ix], P[ix].max(1), P[ix].argmax(1))
        boots = []
        for _ in range(N_CLUSTER_BOOT):
            ix = cluster_indices(r["clip_ids"], r["farms"], rng)
            boots.append((ece(r["raw_P"], ix), ece(r["cal_P"], ix)))
        boots = np.array(boots)
        full = np.arange(len(r["y"]))
        ece_rows.append({"model": m, "target": t,
                         "ece_raw": ece(r["raw_P"], full), "ece_raw_ci_low": np.quantile(boots[:, 0], 0.025),
                         "ece_raw_ci_high": np.quantile(boots[:, 0], 0.975),
                         "ece_nested_calibrated": ece(r["cal_P"], full),
                         "ece_cal_ci_low": np.quantile(boots[:, 1], 0.025),
                         "ece_cal_ci_high": np.quantile(boots[:, 1], 0.975),
                         "change_ci_low": np.quantile(boots[:, 1] - boots[:, 0], 0.025),
                         "change_ci_high": np.quantile(boots[:, 1] - boots[:, 0], 0.975)})
ece_ci_df = pd.DataFrame(ece_rows)
display(cluster_df.round(3))
display(ece_ci_df.round(3))

In [ ]:
TIE_DRAWS = 2000
tie_rows = []
for m in MODELS:
    store = l3[m][0]
    for t in TARGETS:
        r = store[t]
        P, y = r["sel_P"], r["y"]
        conf, pred = P.max(1), P.argmax(1)
        k = int(round(0.80 * len(y)))
        order = np.argsort(-conf, kind="stable")
        cut = conf[order[k - 1]]
        n_tied = int(np.isclose(conf, cut, rtol=0, atol=1e-12).sum())
        reported = mf1(y[order[:k]], pred[order[:k]])
        rng = np.random.default_rng(SEED)
        draws = []
        for _ in range(TIE_DRAWS):
            o = np.lexsort((rng.random(len(conf)), -conf))
            draws.append(mf1(y[o[:k]], pred[o[:k]]))
        tie_rows.append({"model": m, "target": t, "n_tied_at_cutoff": n_tied,
                         "retained_macro_f1_reported": reported,
                         "retained_macro_f1_random_ties_mean": float(np.mean(draws)),
                         "retained_macro_f1_random_ties_min": float(np.min(draws)),
                         "retained_macro_f1_random_ties_max": float(np.max(draws))})
ties_df = pd.DataFrame(tie_rows)
ties_df.round(3)

In [ ]:
discrimination_df.to_csv(OUT_DIR / "discrimination_summary.csv", index=False)
perfarm_df.to_csv(OUT_DIR / "discrimination_perfarm.csv", index=False)
differences_df.to_csv(OUT_DIR / "paired_bootstrap_differences.csv", index=False)
reliability_df.to_csv(OUT_DIR / "reliability_summary.csv", index=False)
curves_df.to_csv(OUT_DIR / "selective_coverage_curves.csv", index=False)
folds_df.to_csv(OUT_DIR / "calibration_selection_by_fold.csv", index=False)
ties_df.to_csv(OUT_DIR / "cutoff_tie_check.csv", index=False)
cluster_df.to_csv(OUT_DIR / "cluster_bootstrap_differences.csv", index=False)
ece_ci_df.to_csv(OUT_DIR / "ece_cluster_intervals.csv", index=False)
with open(OUT_DIR / "comparison_manifest.json", "w", encoding="utf-8") as f:
    json.dump({"build_ts_utc": RUN_TS, "models": {m: [str(p) for p in SOURCES[m][:2]] for m in MODELS},
               "bootstrap": {"n": N_BOOT, "seed": SEED, "unit": "clip, paired across models"},
               "cluster_bootstrap": {"n": N_CLUSTER_BOOT, "seed": SEED, "unit": "recording-session group resampled within farm; Holm-adjusted p-values"},
               "tie_draws": TIE_DRAWS, "coverage_operating_point": 0.80}, f, indent=2)

In [ ]:
pd.set_option("display.width", 250)
display(perfarm_df.pivot_table(index=["target", "heldout_farm"], columns="model", values="macro_f1").round(3))
display(differences_df.round(3))
display(reliability_df.drop(columns=["calibration_method", "selective_score", "learned_threshold_coverage_by_farm"]).round(3))
display(ties_df.round(3))